# nanochat.cpp end to end on Kaggle (1x T4)

Build the CUDA library for the T4 (`sm_75`), prepare ClimbMix and the
`NCTOKEN1` tokenizer, pretrain a small base model, and post-train it with
supervised fine-tuning in one process.

The setup module `notebooks/kaggle_setup.py` prepares the session. The
trial runner `notebooks/lab.py` holds the run logic. The notebook trains one
small base model, evaluates it on the real ClimbMix data, and post-trains it
with supervised fine-tuning. The design is `docs/notebook-workflow.md`. The
runtime uses one T4. The C++ workflow has no distributed mode
(`docs/host-portability.md`). The Python surface is `docs/python.md`. The
evaluation mirrors `docs/eval.md`. The post-training path mirrors
`docs/post-training.md`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

from kaggle_secrets import UserSecretsClient

# The Hugging Face token lets the data download run without a prompt.
try:
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret; the download may be rate limited")

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")
BASE = Path("/kaggle/working/nanochat-cache")

# Clone the repository, then make its packages importable.
if not REPO.is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
sys.path[:0] = [str(REPO / "python"), str(REPO / "notebooks")]
print("repository:", REPO)


## Build

The loader cell clones the repository and makes it importable.
`kaggle_setup.setup` does the rest: fetch, `tools/kaggle/bootstrap.sh`, the
host environment, and the T4 library build. The session procedure is
`docs/host-portability.md`, section 8.


In [ ]:
import kaggle_setup

host = kaggle_setup.setup(url=REPO_URL, dest=REPO, arch="sm_75")
print("library:", host.library)
print("build key:", host.build_key)

import nanochat_cpp as nc

devices = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total",
     "--format=csv,noheader"], capture_output=True,
    text=True).stdout.strip()
print(devices)


In [ ]:
from pi_agent import run_pi

PI_PROVIDER = os.environ.get("PI_AGENT_PROVIDER", "opencode-go")
PI_MODEL = os.environ.get("PI_AGENT_MODEL", "deepseek-v4.1-flash")
PI_PROMPT = (
    "Inspect this repository and report a concise status. "
    "Do not edit files.")
run_pi(PI_PROMPT, cwd=REPO,
       provider=PI_PROVIDER, model=PI_MODEL)


## Data and tokenizer

One ClimbMix shard trains the tokenizer and the base model. The validation
shard is the reference split.


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U",
                "huggingface_hub"], check=True)

from huggingface_hub import snapshot_download

DATA = BASE / "base_data_climbmix"
TRAIN_SHARD = "shard_00000.parquet"   # one shard is enough for a small model
VAL_SHARD = "shard_06542.parquet"     # the reference validation split

if not (DATA / VAL_SHARD).is_file():
    snapshot_download(repo_id="karpathy/climbmix-400b-shuffle",
                      repo_type="dataset", local_dir=str(DATA),
                      allow_patterns=[TRAIN_SHARD, VAL_SHARD],
                      token=os.environ.get("HF_TOKEN"))

print("train:", (DATA / TRAIN_SHARD).name,
      "validation:", (DATA / VAL_SHARD).name)


In [ ]:
artifact = BASE / "tokenizer" / "tokenizer.nctoken"

if not artifact.is_file():
    artifact.parent.mkdir(parents=True, exist_ok=True)
    # The orchestration layer calls tools/nanochat for the build and the run.
    nc.toolchain.build(["//src/tokenizer:tok_train_main"])
    nc.toolchain.run("t0-cpu", [
        "bazel-bin/src/tokenizer/tok_train_main",
        "--parquet", f"{DATA}/{TRAIN_SHARD}",
        "--vocab-size", "32768",
        "--out", str(artifact),
    ])

tokenizer = nc.Tokenizer.load(str(artifact))
assert tokenizer.decode(tokenizer.encode("The capital of France is")) == \
    "The capital of France is", "the tokenizer round trip failed"
print("tokenizer:", tokenizer.vocab_size, "tokens")


## Pretrain

The plan sets the model shape, the horizon, and the scaled rates. One optimizer
step consumes one micro batch.

The trial runner `notebooks/lab.py` runs the plan and writes a run directory
under `RUNS`. The shared inputs live in one `Prefix`, so the build, the data,
and the tokenizer run once. The run-directory contract is
`docs/notebook-workflow.md`.


In [ ]:
import lab

RUNS = Path("/kaggle/working/runs")

prefix = lab.Prefix(
    tokenizer=tokenizer,
    train_parquet=f"{DATA}/{TRAIN_SHARD}",
    val_parquet=f"{DATA}/{VAL_SHARD}",
    data_seed=7)

# The demo always runs, so the fine-tuning cell has a live model.
demo = lab.run_trial(
    lab.Trial("demo", depth=8, seq_len=512, device_batch_size=8,
              total_batch_size=4096, num_iterations=50),
    prefix, RUNS, keep_model=True, force=True, build_key=host.build_key,
    progress=lab.ProgressReporter())

model, plan = demo.model, demo.plan
print("device:", model.device, "parameters:", f"{model.param_count():,}")
print(f"final loss {demo.summary['final_loss']:.4f}  "
      f"bpb {demo.summary['bpb']:.4f}  "
      f"{demo.summary['tokens_per_sec']:.0f} tokens/s  "
      f"mfu {demo.summary['mfu'] * 100:.2f}%  "
      f"revision {demo.summary['revision']}")


## Pretraining metrics

The loss, the throughput, and the MFU from the run above.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

frame = pd.DataFrame(demo.metrics).set_index("step")
axes = frame[["loss", "tokens_per_sec", "mfu"]].plot(
    subplots=True, figsize=(11, 5), grid=True,
    title=["Pretraining loss", "Throughput", "MFU"])
axes[0].set_ylabel("cross-entropy loss")
axes[1].set_ylabel("tokens / second")
axes[2].set_ylabel("model FLOPs utilization")
axes[0].figure.suptitle(f"nanochat.cpp pretraining on {model.device}")
plot_path = Path("/kaggle/working/pretraining_metrics.png")
axes[0].figure.savefig(plot_path, dpi=120)
print("saved:", plot_path)
plt.show()


## Base evaluation

The reference `scripts/base_eval.py` reports bits per byte on the real
ClimbMix train and validation splits. This section mirrors that measurement
with `nc.evaluate`, which reproduces `evaluate_bpb` (`docs/eval.md`, section
4.1). The reference uses `40 * 524288` tokens per split. The notebook uses a
smaller budget, so the session finishes. The sample prompts are the reference
list. CORE is not implemented yet (`docs/eval.md`, section 9).


In [ ]:
# Base evaluation on the real ClimbMix splits, like scripts/base_eval.py.
EVAL_SPLIT_TOKENS = 4 * 524288   # the reference default is 40 * 524288

def base_bpb_for(parquet, split):
    report = nc.evaluate(model, parquet=str(parquet), tokenizer=tokenizer,
                         tokens=EVAL_SPLIT_TOKENS,
                         batch=plan.device_batch_size, seed=43)
    print(f"base {split:>5} bpb: {report.bpb:.4f}")
    return report.bpb

base_bpb = {"train": base_bpb_for(DATA / TRAIN_SHARD, "train"),
            "val": base_bpb_for(DATA / VAL_SHARD, "val")}
print(f"base validation bits per byte: {base_bpb['val']:.4f}")

# The reference sample prompts, greedy (scripts/base_eval.py).
SAMPLE_PROMPTS = [
    "The capital of France is",
    "The chemical symbol of gold is",
    "If yesterday was Friday, then tomorrow will be",
    "The opposite of hot is",
    "The planets of the solar system are:",
    "My favorite color is",
    "If 5*x + 3 = 13, then x is",
]

bos = tokenizer.get_bos_token_id()
for prompt in SAMPLE_PROMPTS:
    prompt_ids = ([bos] if bos >= 0 else []) + tokenizer.encode(prompt)
    row = model.generate(prompt_ids, max_tokens=16, temperature=0.0)[0]
    generated = [token for token, keep in zip(row.tokens, row.mask) if keep]
    print("-" * 40)
    print(prompt + tokenizer.decode(generated))


## Post-train: supervised fine-tuning

The base model becomes a chat model on the reference conversation mixture:
SmolTalk (`HuggingFaceTB/smol-smoltalk`), MMLU (`cais/mmlu`), and GSM8K
(`openai/gsm8k`). `nanochat_cpp.sft_data` owns that mixture. The notebook uses a
small slice of it, so the run stays inside the session budget. When the hub is
unreachable, `notebooks/kaggle_demo.py` falls back to a small offline set.

`nc.render_conversation` returns the token ids and a loss mask. `mask = 1` marks
the assistant text, the assistant end token, and the python tool-call tokens.
`mask = 0` marks the BOS, the user text, and the tool outputs. The packer uses
the reference BOS-aligned best-fit algorithm. It pads a partial row with BOS
instead of cropping, and it masks the padding. The packer then shifts the mask
by one position, because position `j` predicts token `j + 1`. A masked target
becomes `-1`, and the loss divides by the valid tokens only. The packer, the
data helper, and the completion helper live in `notebooks/kaggle_demo.py`.


In [ ]:
from kaggle_demo import build_sft_conversations, complete

SFT_SEQ = 128
SFT_STEPS = 40
SFT_ROWS = 64

# Real reference data: SmolTalk, MMLU, and GSM8K. The helper returns the
# offline DEMO_CONVERSATIONS set when the hub download fails.
sft_conversations = build_sft_conversations(limit=SFT_ROWS)
sft_rows = nc.sft_data.pack_rows(sft_conversations, tokenizer, SFT_SEQ)
print(f"{len(sft_rows)} packed rows of {SFT_SEQ} tokens "
      f"from {len(sft_conversations)} conversations")

# Demo fine-tuning rates. They mirror the reference SFT starting point.
sft_optimizer = nc.Optimizer(
    model,
    embedding_lr=0.3, unembedding_lr=0.008, matrix_lr=0.02, scalar_lr=0.5,
    weight_decay=0.0,
    num_iterations=SFT_STEPS, warmup_steps=2, warmdown_ratio=0.5,
    final_lr_frac=0.0)

base_completion = complete(model, tokenizer, "The capital of France is")

sft_losses = []
for step in range(1, SFT_STEPS + 1):
    tokens, targets = sft_rows[(step - 1) % len(sft_rows)]
    sft_optimizer.zero_grad()
    loss = model.forward_loss(tokens, targets, batch=1, seq=SFT_SEQ)
    model.backward()
    sft_optimizer.step(step)
    sft_losses.append(loss)
    if step == 1 or step % 10 == 0 or step == SFT_STEPS:
        print(f"sft step {step:3d}/{SFT_STEPS}  loss {loss:.4f}")

sft_completion = complete(model, tokenizer, "The capital of France is")

sft_checkpoint = Path("/kaggle/working/chatsft.nchkpt01")
model.save(str(sft_checkpoint))
print("base completion:", base_completion)
print("sft completion: ", sft_completion)
print(f"chatsft checkpoint: {sft_checkpoint.stat().st_size:,} bytes")
early = sum(sft_losses[:5]) / 5
late = sum(sft_losses[-5:]) / 5
assert late < early, "the fine-tuning loss did not decrease"

# Reload the checkpoint and score it.
reloaded = nc.Model(plan.config, device="cuda", seed=0)
reloaded.load(str(sft_checkpoint))
batch_tokens = list(range(plan.config.seq_len))
reload_loss = reloaded.forward_loss(batch_tokens, batch_tokens[1:] + [0],
                                    batch=1, seq=plan.config.seq_len)
assert reload_loss == reload_loss, "the reloaded model scores a finite loss"

base_bpb = demo.summary["bpb"]
sft_bpb = nc.evaluate(
    model, parquet=str(DATA / VAL_SHARD), tokenizer=tokenizer,
    tokens=plan.device_batch_size * plan.config.seq_len * 4,
    batch=plan.device_batch_size, seed=43).bpb
print(f"base {base_bpb:.4f} bits per byte, post-trained {sft_bpb:.4f}")
print("final checks passed")


## Chat evaluation

The reference `scripts/chat_eval.py` scores the post-trained model on real
tasks: ARC-Easy, ARC-Challenge, MMLU, GSM8K, and HumanEval. This section
mirrors that run with `nc.chat.ChatEvaluator` (`docs/eval.md`, section 5). The
evaluator isolates each task. A missing dataset or a missing reference package
becomes a skipped row. `chat_core` is the mean centered accuracy over the
tasks, and it is `None` when a task is skipped.


In [ ]:
# The reference chat tasks on the real task data (scripts/chat_eval.py).
EVAL_TASKS = ["ARC-Easy", "ARC-Challenge", "MMLU", "GSM8K", "HumanEval"]
EVAL_PROBLEMS = 20   # a small slice; the full reference run is larger

chat_report = nc.chat.ChatEvaluator(model, tokenizer=tokenizer).run(
    tasks=EVAL_TASKS, max_problems=EVAL_PROBLEMS, seed=0)

chat_rows = []
for report in chat_report.tasks:
    status = "skipped" if report.skipped else f"{report.accuracy:.3f}"
    print(f"{report.name:>14}  {status:>8}  "
          f"({report.correct}/{report.total})")
    chat_rows.append({"task": report.name, "accuracy": report.accuracy,
                      "correct": report.correct, "total": report.total,
                      "baseline": report.baseline,
                      "skipped": report.skipped, "error": report.error})

print("categorical core:", chat_report.categorical_core)
print("chat core:", chat_report.chat_core)
display(pd.DataFrame(chat_rows).set_index("task"))
